# Security of Supply Analysis

In this tutorial, we will analyse the security of supply of different scenarios using the 3-node model we have been working on in the previous tutorial. In this setting, we will focus on dispatch optimisation under different stress events and evaluate the performance of the system using different metrics.


:::{note}
If you have not yet set up Python on your computer, you can execute this tutorial in your browser via [Google Colab](https://colab.research.google.com/). Download the `.ipynb` file using the download button on the top right corner and import it in [Google Colab](https://colab.research.google.com/).

Then install the following packages by executing the following command in a Jupyter cell at the top of the notebook.

```sh
!pip install pypsa plotly"<6"
```
:::

We start by importing the necessary libaries and loading the base network. This network is amended by a load shedding generator at each node and we also resample the time series to 3h resolution to speed up the optimisation.

In [109]:
import pypsa
import pandas as pd

import plotly.io as pio
import plotly.offline as py

pd.options.plotting.backend = "plotly"
pypsa.options.params.optimize.log_to_console = False

In [110]:
url = "https://tubcloud.tu-berlin.de/s/M2QqWfES4Md8Xoe/download/phl-network.nc"
base = pypsa.Network(url)

INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads, storage_units


In [111]:
base.add(
    "Generator",
    base.buses.index + " load-shedding",
    bus=base.buses.index,
    carrier="load-shedding",
    marginal_cost=2000,
    p_nom=base.loads_t.p_set.max().max(),
)
base.add("Carrier", "load-shedding", color="crimson")


In [112]:
base = base.cluster.temporal.resample("4h")

In [113]:
n0 = base.copy()
n0.name = "base"

## Events

### Power Plant Decommissioning

We decommission 2 GW of coal generation capacity in Luzon and all coal generation capacity in Mindanao.

In [114]:
n1 = base.copy()
n1.name = "coal-decommissioning"
n1.generators.loc["Luzon coal-subcritical", "p_nom"] -= 2000
n1.generators.loc["Mindanao coal-subcritical", "p_nom"] -= 836

### Power Plant Outage

We simulate the outage of Luzon's supercritical coal power plants in all of February.

In [115]:
n2 = base.copy()
n2.name = "coal-generator-outage"

n2.generators_t.p_max_pu.loc["2025-02", "Luzon coal-supercritical"] = 0
n2.generators_t.p_max_pu.fillna(1, inplace=True)

### Interconnector Outage

We simulate the outage of the interconnectors between all three regions throughout the full year (i.e. no power can be transmitted between the regions).

In [116]:
# temporary outage of all interconnectors for 4 weeks in September
n3 = base.copy()
n3.name = "interconnector-outage"

n3.links.p_max_pu = 0
n3.links.p_min_pu = 0

### Volcano Eruption

We simulate a volcano eruption that reduces solar photovoltaics generation by 80% for the first half of the year.

In [117]:
n4 = base.copy()
n4.name = "volcano-eruption"

pv_gens = n4.generators.carrier == "photovoltaic"

n4.generators_t.p_max_pu.loc[
    "2025-02":"2025-06",
    pv_gens,
] *= 0.2

### Price Shock in Oil and Gas

We simulate a price shock in oil and gas (e.g. due to a geopolitical crisis) that doubles the fuel costs of oil and gas generators for the full year.

In [118]:
n5 = base.copy()
n5.name = "price-shock-oil-gas"

selection = n5.generators.carrier.str.contains("oil|gas").index
n5.generators.loc[selection, "marginal_cost"] *= 2

### Demand Increase

We simulate a demand increase of 20% across all three regions for the full year (e.g. due to economic growth).

In [119]:
n6 = base.copy()
n6.name = "demand-increase"

n6.loads_t.p_set *= 1.2

### Drought

We simulate a drought event that reduces hydro generation by 50% for the full year.

In [120]:
n7 = base.copy()
n7.name = "drought"

In [121]:
# drought reduces hydro generation by 50%
hydro = n7.generators.carrier == "hydro-reservoir-and-run-of-river"
n7.generators_t.p_max_pu.loc[:, hydro] *= 0.5

### Coal Import Limitation

We simulate a coal usage restriction that limits the total coal consumption across the system to 120 TWh thermal for the full year (e.g. due to a coal import limitation).

Note, that we need to factor in the efficiency of the coal generators to calculate the thermal energy consumption from the electrical energy generation.

This is a constraint that is not directly supported by PyPSA, so we need to add it manually to the model as a **custom constraint** after the model has been created but before it is optimised.

In [ ]:
n8 = base.copy()
n8.name = "coal-import-limit"

n8.optimize.create_model()
coal_i = n8.generators.loc[n8.generators.carrier.str.contains("coal")].index
lhs = (
    n8.model["Generator-p"]
    .loc[:, coal_i]
    .mul(n.snapshot_weightings.generators)
    .sum(dim="snapshot")
    .div(n8.generators.loc[coal_i, "efficiency"])
    .sum()
)
n8.model.add_constraints(lhs <= 80e6, name="Generator-coal_limit");

/tmp/ipykernel_979235/2300290386.py:4: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.



### Reserve Constraints

In a similar way, we can also add constraints to ensure that there is sufficient reserve capacity available in the system to cover for unexpected events. For example, we can require that there is always at least 20% of the hour's demand or at least 3 GW of reserve capacity available.

In [123]:
n9 = base.copy()
n9.name = "reserve-constraint"

n9.optimize.create_model()

regex = "coal|gas|geothermal|oil|bioenergy"
reserve_i = n9.generators.loc[n9.generators.carrier.str.contains(regex)].index

/tmp/ipykernel_979235/4234672280.py:4: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.



We use the dispatch of firm generators (i.e. generators that are not renewable):

In [124]:
p_reserve = n9.model["Generator-p"].loc[:, reserve_i].sum(dim="name")
p_reserve

LinearExpression [snapshot: 2190]:
----------------------------------
[2025-01-01 08:00:00]: +1 Generator-p[2025-01-01 08:00:00, Mindanao coal-ultrasupercritical] + 1 Generator-p[2025-01-01 08:00:00, Mindanao geothermal-unspecified] + 1 Generator-p[2025-01-01 08:00:00, Mindanao bioenergy-unspecified] ... +1 Generator-p[2025-01-01 08:00:00, Visayas coal-circulating-fluidized-bed] + 1 Generator-p[2025-01-01 08:00:00, Visayas gas-combined-cycle] + 1 Generator-p[2025-01-01 08:00:00, Visayas gas-ccs]
[2025-01-01 12:00:00]: +1 Generator-p[2025-01-01 12:00:00, Mindanao coal-ultrasupercritical] + 1 Generator-p[2025-01-01 12:00:00, Mindanao geothermal-unspecified] + 1 Generator-p[2025-01-01 12:00:00, Mindanao bioenergy-unspecified] ... +1 Generator-p[2025-01-01 12:00:00, Visayas coal-circulating-fluidized-bed] + 1 Generator-p[2025-01-01 12:00:00, Visayas gas-combined-cycle] + 1 Generator-p[2025-01-01 12:00:00, Visayas gas-ccs]
[2025-01-01 16:00:00]: +1 Generator-p[2025-01-01 16:00:00, Mindanao 

as well as the capacity of firm generators to calculate the available reserve capacity in each hour and add a constraint to ensure that this is always above the required reserve margin.:

In [125]:
p_nom_reserve = n9.generators.p_nom.loc[reserve_i].sum()
p_nom_reserve

np.float64(22957.079999999998)

The reserve requirement we calculate as the maximum of 20% of the demand or 3 GW:

In [126]:
reserve_required = n9.loads_t.p_set.sum(axis=1).mul(0.2).clip(lower=3000)
reserve_required.plot()

This is added as a constraint to the model:

In [127]:
expr = p_nom_reserve - p_reserve >= reserve_required
n9.model.add_constraints(expr, name="reserve_constraints");

### Compound Stress Events

**You decide which events to combine.**


In [128]:
n10 = base.copy()
n10.name = "compound-event"

## Solving Scenarios

Now, we can solve the different scenarios. Note that the cases where we have added custom constraints do not require to rebuild the model, but just require it to be solved, so the syntax is slightly different for these cases.

In [129]:
for n in [n0, n1, n2, n3, n4, n5, n6, n7, n10]:
    n.optimize()

n8.optimize.solve_model()
n9.optimize.solve_model()

/tmp/ipykernel_979235/1324107225.py:2: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00, 224.57it/s]
INFO:linopy.io: Writing time: 0.34s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 155490 primals, 330690 duals
Objective: 4.35e+09
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-uppe

('ok', 'optimal')

## Network Collections

PyPSA has a [`pypsa.NetworkCollection`](https://docs.pypsa.org/latest/user-guide/collection/) class that provides a convenient way to manage and analyse multiple networks simultaneously, access their combined data, and generate combined statistics and plots. It uses the names of the networks as indices.

In [130]:
nc = pypsa.NetworkCollection([n0, n1, n2, n3, n4, n5, n6, n7, n8, n9, n10])

## Evaluation Metrics

### Operational Cost

As absolute values by technology, as total, as relative difference, or as absolute difference to the base case.

In [131]:
opex = nc.statistics.opex().div(1e6).round(1).droplevel("component").unstack("network")

In [132]:
opex.sum()

network
base                     4350.5
coal-decommissioning     4449.5
coal-generator-outage    4361.2
coal-import-limit        6185.5
compound-event           4350.5
demand-increase          5776.3
drought                  4686.6
interconnector-outage    4389.2
price-shock-oil-gas      8699.2
reserve-constraint       4350.5
volcano-eruption         4451.9
dtype: float64

In [133]:
opex.sum() / opex.sum().min()

network
base                     1.000000
coal-decommissioning     1.022756
coal-generator-outage    1.002459
coal-import-limit        1.421791
compound-event           1.000000
demand-increase          1.327732
drought                  1.077255
interconnector-outage    1.008896
price-shock-oil-gas      1.999586
reserve-constraint       1.000000
volcano-eruption         1.023308
dtype: float64

In [134]:
opex.sum() - opex.sum().min()

network
base                        0.0
coal-decommissioning       99.0
coal-generator-outage      10.7
coal-import-limit        1835.0
compound-event              0.0
demand-increase          1425.8
drought                   336.1
interconnector-outage      38.7
price-shock-oil-gas      4348.7
reserve-constraint          0.0
volcano-eruption          101.4
dtype: float64

### Electricity Mix

In [135]:
energy = (
    nc.statistics.energy_balance()
    .div(1e6)
    .round(1)
    .groupby(["network", "carrier"])
    .sum()
    .unstack("network")
    .drop("AC")
)
energy.T.plot(
    kind="bar", barmode="stack", labels=dict(value="Electricity Supply [TWh]")
)

In [136]:
nc["demand-increase"].statistics.energy_balance.iplot()

### Prices

We can look at average prices or just the frequency of hours with very high prices (e.g. above 100 $/MWh).

In [137]:
nc.statistics.prices.iplot()

In [138]:
(nc.buses_t.marginal_price > 100).sum().unstack("network").div(8760 / 3 / 100).round(1)

network,base,coal-decommissioning,coal-generator-outage,interconnector-outage,volcano-eruption,price-shock-oil-gas,demand-increase,drought,coal-import-limit,reserve-constraint,compound-event
name,,,,,,,,,,,
Luzon,0.0,0.0,0.0,0.0,0.0,75.0,0.0,0.0,75.0,0.0,0.0
Visayas,0.0,0.0,0.0,0.0,0.0,34.0,1.8,0.0,75.0,0.0,0.0
Mindanao,0.0,6.3,0.0,0.0,0.0,34.0,1.8,0.0,75.0,0.0,0.0


### Expected Energy Not Served

The expected energy not served (EENS) is a common reliability metric that quantifies the expected amount of energy demand that cannot be met due to supply shortages. It is calculated as the energy used by the load shedding generators in our model.

In [ ]:
# eens
(
    nc.snapshot_weightings.generators
    @ nc.generators_t.p.loc[:, nc.generators.carrier == "load-shedding"]
).unstack("network")

network,base,coal-decommissioning,coal-generator-outage,interconnector-outage,volcano-eruption,price-shock-oil-gas,demand-increase,drought,coal-import-limit,reserve-constraint,compound-event
name,,,,,,,,,,,
Luzon load-shedding,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Visayas load-shedding,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Mindanao load-shedding,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Loss of Load Probability

The loss of load probability (LOLP) is a reliability metric that quantifies the probability that the system will not be able to meet the demand in a given hour. It is calculated as the number of hours with load shedding divided by the total number of hours.

In [ ]:
# LOLE
(
    nc.generators_t.p.loc[:, nc.generators.carrier == "load-shedding"] > 0.1
).sum().unstack("network") / (8760 / 3)

network,base,coal-decommissioning,coal-generator-outage,interconnector-outage,volcano-eruption,price-shock-oil-gas,demand-increase,drought,coal-import-limit,reserve-constraint,compound-event
name,,,,,,,,,,,
Luzon load-shedding,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Visayas load-shedding,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Mindanao load-shedding,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Maximum Loss of Load

The maximum loss of load (MLOL) is a reliability metric that quantifies the maximum amount of power demand that cannot be met in any given hour. It is calculated as the maximum power used by the load shedding generators in our model.

In [141]:
n.generators_t.p.loc[:, n.generators.carrier == "load-shedding"].max()

name
Luzon load-shedding      -0.0
Visayas load-shedding    -0.0
Mindanao load-shedding   -0.0
dtype: float64

### Reserve Margin

The reserve margin is a static reliability metric that quantifies the amount of reserve capacity available in the system with the peak demand subtracted.

In [142]:
(
    n.generators.filter(regex="coal|geothermal|oil|gas|bioenergy", axis=0)
    .groupby("bus")
    .p_nom.sum()
    - n.loads_t.p_set.max()
)


bus
Luzon       2958.1000
Mindanao     553.1075
Visayas      681.0400
dtype: float64

## Exercises

**Task 1:** For scenario `n10`, create different combinations of the above events and analyse the results. For example, you could combine a drought with a coal import limitation, or a power plant outage with a price shock in oil and gas.

**Task 2:** Change the capacities of the system (e.g. increase solar capacity, add battery storage or increase gas capacity). Repeat the analysis for the different scenarios and compare the results. Can you identify effective strategies for improving the security of supply? Which changes cause more load shedding?